In [1]:
# Install libraries
!pip -q install numpy pandas torch matplotlib scikit-learn

In [2]:
# Imports
import os
import math
import random
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from google.colab import files

warnings.filterwarnings("ignore")

In [3]:
# Config
SUBSET = "FD001"   # change to FD002 / FD003 / FD004 if needed
WINDOW = 30
STRIDE = 5
EPOCHS = 20
BATCH = 512
HIDDEN = 96
LAYERS = 2
DROPOUT = 0.2
LR = 1e-3
CAP = 125
SEED = 42

SENSOR_COUNT = 21
OP_COUNT = 3

BASE_COLS = ["unit", "cycle"]
OP_COLS = [f"op{i}" for i in range(1, OP_COUNT + 1)]
SENSOR_COLS = [f"s{i}" for i in range(1, SENSOR_COUNT + 1)]
ALL_COLS = BASE_COLS + OP_COLS + SENSOR_COLS

ROOT_DIR = Path("/content")
DATA_DIR = ROOT_DIR / "data"
RESULTS_DIR = ROOT_DIR / "results"

DATA_DIR.mkdir(exist_ok=True, parents=True)
RESULTS_DIR.mkdir(exist_ok=True, parents=True)


In [4]:
# Seed setup
def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("PyTorch version:", torch.__version__)

Using device: cuda
PyTorch version: 2.10.0+cu128


In [5]:
# Upload txt files directly
print("Upload all needed dataset txt files now.")
print("Example: train_FD001.txt, test_FD001.txt, RUL_FD001.txt")
uploaded = files.upload()

if len(uploaded) == 0:
    raise FileNotFoundError("No files were uploaded.")

for fname, filedata in uploaded.items():
    save_path = DATA_DIR / fname
    with open(save_path, "wb") as f:
        f.write(filedata)

print("\nUploaded files saved in:", DATA_DIR)
for f in DATA_DIR.iterdir():
    if f.is_file():
        print(f.name)


Upload all needed dataset txt files now.
Example: train_FD001.txt, test_FD001.txt, RUL_FD001.txt


Saving RUL_FD001.txt to RUL_FD001.txt
Saving RUL_FD002.txt to RUL_FD002.txt
Saving RUL_FD003.txt to RUL_FD003.txt
Saving RUL_FD004.txt to RUL_FD004.txt
Saving test_FD001.txt to test_FD001.txt
Saving test_FD002.txt to test_FD002.txt
Saving test_FD003.txt to test_FD003.txt
Saving test_FD004.txt to test_FD004.txt
Saving train_FD001.txt to train_FD001.txt
Saving train_FD002.txt to train_FD002.txt
Saving train_FD003.txt to train_FD003.txt
Saving train_FD004.txt to train_FD004.txt

Uploaded files saved in: /content/data
train_FD004.txt
train_FD001.txt
train_FD003.txt
RUL_FD004.txt
RUL_FD003.txt
test_FD002.txt
test_FD004.txt
test_FD003.txt
RUL_FD002.txt
RUL_FD001.txt
train_FD002.txt
test_FD001.txt


In [24]:
# READ DATA

def read_txt(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None, engine="python")
    if df.shape[1] > len(ALL_COLS):
        df = df.iloc[:, :len(ALL_COLS)]
    df.columns = ALL_COLS
    return df

def load_data(data_dir: Path, subset: str):
    train_path = data_dir / f"train_{subset}.txt"
    test_path = data_dir / f"test_{subset}.txt"
    rul_path = data_dir / f"RUL_{subset}.txt"

    if not train_path.exists():
        raise FileNotFoundError(f"Missing file: {train_path.name}")
    if not test_path.exists():
        raise FileNotFoundError(f"Missing file: {test_path.name}")
    if not rul_path.exists():
        raise FileNotFoundError(f"Missing file: {rul_path.name}")

    print("\nUsing files:")
    print("Train:", train_path)
    print("Test :", test_path)
    print("RUL  :", rul_path)

    train = read_txt(train_path)
    test = read_txt(test_path)
    rul = pd.read_csv(rul_path, header=None, sep=r"\s+", engine="python")
    rul = rul.iloc[:, :1]
    rul.columns = ["RUL_end"]

    return train, test, rul

def add_train_rul(df: pd.DataFrame, cap: int) -> pd.DataFrame:
    max_cycle = df.groupby("unit")["cycle"].max().rename("max_cycle")
    df = df.join(max_cycle, on="unit")
    df["RUL"] = df["max_cycle"] - df["cycle"]
    df.drop(columns=["max_cycle"], inplace=True)
    df["RUL"] = df["RUL"].clip(upper=cap)
    return df

def add_test_rul(df: pd.DataFrame, rul_df: pd.DataFrame, cap: int) -> pd.DataFrame:
    max_cycle = df.groupby("unit")["cycle"].max().rename("max_cycle")
    df = df.join(max_cycle, on="unit")

    units = sorted(df["unit"].unique())
    rul_map = {units[i]: int(rul_df.iloc[i, 0]) for i in range(len(units))}

    df["RUL_end"] = df["unit"].map(rul_map)
    df["RUL"] = (df["max_cycle"] - df["cycle"]) + df["RUL_end"]

    df.drop(columns=["max_cycle"], inplace=True)
    df.drop(columns=["RUL_end"], inplace=True)
    df["RUL"] = df["RUL"].clip(upper=cap)
    return df


# PLOTTING HELPERS

def save_rul_distribution(df: pd.DataFrame, results_dir: Path) -> None:
    plt.figure(figsize=(8, 5))
    plt.hist(df["RUL"], bins=50)
    plt.title("RUL Distribution")
    plt.xlabel("Remaining Useful Life (cycles)")
    plt.ylabel("Number of Records")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(results_dir / "rul_distribution.png", dpi=300)
    plt.close()

def save_sensor_trend(df: pd.DataFrame, results_dir: Path, sensor_name: str = "s1") -> None:
    unit_id = df["unit"].iloc[0]
    sample = df[df["unit"] == unit_id].copy()

    plt.figure(figsize=(8, 5))
    plt.plot(sample["cycle"], sample[sensor_name])
    plt.title(f"{sensor_name.upper()} Degradation Trend for Engine {unit_id}")
    plt.xlabel("Cycle Number")
    plt.ylabel(f"{sensor_name.upper()} Sensor Reading")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(results_dir / "sensor_trend.png", dpi=300)
    plt.close()

def save_training_loss_plot(history, model_name: str, results_dir: Path) -> None:
    epochs = np.arange(1, len(history) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history, marker="o")
    plt.title(f"{model_name} Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Mean Squared Error (MSE) Loss")
    plt.xticks(epochs)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(results_dir / f"{model_name}_loss.png", dpi=300)
    plt.close()

def save_prediction_scatter(y_true, y_pred, model_name: str, results_dir: Path) -> None:
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.8)

    min_val = min(np.min(y_true), np.min(y_pred))
    max_val = max(np.max(y_true), np.max(y_pred))
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

    plt.title(f"{model_name}: Predicted RUL vs True RUL")
    plt.xlabel("True RUL (cycles)")
    plt.ylabel("Predicted RUL (cycles)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(results_dir / f"{model_name}_scatter.png", dpi=300)
    plt.close()

def save_error_distribution(y_true, y_pred, model_name: str, results_dir: Path) -> None:
    errors = y_pred - y_true

    plt.figure(figsize=(8, 5))
    plt.hist(errors, bins=30)
    plt.title(f"{model_name} Prediction Error Distribution")
    plt.xlabel("Prediction Error = Predicted RUL - True RUL (cycles)")
    plt.ylabel("Number of Test Engines")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(results_dir / f"{model_name}_error.png", dpi=300)
    plt.close()


# EDA

def exploratory_analysis(df: pd.DataFrame, results_dir: Path) -> None:
    print("\n===== DATASET OVERVIEW =====")
    print("Shape:", df.shape)
    print("Number of Engines:", df["unit"].nunique())

    summary = df.describe()
    summary.to_csv(results_dir / "statistical_summary.csv")

    save_rul_distribution(df, results_dir)
    save_sensor_trend(df, results_dir, sensor_name="s1")


# SEQUENCE BUILDER
def build_sequences(df: pd.DataFrame, features, window: int, stride: int):
    X, y = [], []
    df = df.sort_values(["unit", "cycle"])

    for uid, g in df.groupby("unit"):
        g = g.reset_index(drop=True)

        if len(g) < window:
            continue

        feats = g[features].values.astype(np.float32)
        rul = g["RUL"].values.astype(np.float32)

        for i in range(0, len(g) - window + 1, stride):
            X.append(feats[i:i + window])
            y.append(rul[i + window - 1])

    return np.array(X), np.array(y)

def last_windows(df: pd.DataFrame, features, window: int):
    X, y, engines = [], [], []

    for uid, g in df.groupby("unit"):
        g = g.reset_index(drop=True)
        feats = g[features].values.astype(np.float32)

        if len(g) < window:
            pad = np.repeat(feats[:1], window - len(g), axis=0)
            win = np.vstack([pad, feats])
        else:
            win = feats[-window:]

        X.append(win)
        y.append(g["RUL"].iloc[-1])
        engines.append(uid)

    return np.array(X), np.array(y), engines


# DATASET

class RULDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# MODELS
class LSTMModel(nn.Module):
    def __init__(self, input_size: int, hidden: int, layers: int, dropout: float):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1]).squeeze()

class GRUModel(nn.Module):
    def __init__(self, input_size: int, hidden: int, layers: int, dropout: float):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        _, h = self.gru(x)
        return self.fc(h[-1]).squeeze()

class CNNModel(nn.Module):
    def __init__(self, input_size: int, hidden: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_size, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, hidden, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.squeeze(-1)
        return self.fc(x).squeeze()


# METRICS

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2_score_custom(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

def nasa_score(y_true, y_pred):
    d = y_pred - y_true
    score = 0.0
    for di in d:
        if di < 0:
            score += math.exp(-di / 13) - 1
        else:
            score += math.exp(di / 10) - 1
    return score


# TRAINING

def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        preds = model(x)
        loss = loss_fn(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


# LOAD + PREPROCESS DATA

train_df, test_df, rul_df = load_data(DATA_DIR, SUBSET)

train_df = add_train_rul(train_df, CAP)
test_df = add_test_rul(test_df, rul_df, CAP)

print("\nTrain shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("RUL shape  :", rul_df.shape)

display(train_df.head())
display(test_df.head())
display(rul_df.head())

exploratory_analysis(train_df, RESULTS_DIR)

features = OP_COLS + SENSOR_COLS

nunique = train_df[features].nunique()
drop_cols = nunique[nunique <= 1].index.tolist()
if len(drop_cols) > 0:
    print("Dropping constant columns:", drop_cols)
    features = [c for c in features if c not in drop_cols]

scaler = StandardScaler()
scaler.fit(train_df[features])

train_df[features] = scaler.transform(train_df[features])
test_df[features] = scaler.transform(test_df[features])

X_train, y_train = build_sequences(train_df, features, WINDOW, STRIDE)
X_test, y_test, engines = last_windows(test_df, features, WINDOW)

print("\nX_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape :", X_test.shape)
print("y_test shape :", y_test.shape)

train_loader = DataLoader(
    RULDataset(X_train, y_train),
    batch_size=BATCH,
    shuffle=True
)


# TRAIN MODELS

models = {
    "LSTM": LSTMModel(len(features), HIDDEN, LAYERS, DROPOUT).to(device),
    "GRU": GRUModel(len(features), HIDDEN, LAYERS, DROPOUT).to(device),
    "CNN": CNNModel(len(features), HIDDEN).to(device)
}

results = []
predictions_dict = {}

for model_name, model in models.items():
    print(f"\n===== Training {model_name} =====")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    history = []

    for epoch in range(EPOCHS):
        epoch_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
        history.append(epoch_loss)
        print(f"{model_name} | Epoch {epoch + 1:02d}/{EPOCHS} | Training Loss: {epoch_loss:.4f}")

    save_training_loss_plot(history, model_name, RESULTS_DIR)

    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

    preds = np.clip(preds, 0, CAP)

    rmse_val = rmse(y_test, preds)
    mae_val = mae(y_test, preds)
    r2_val = r2_score_custom(y_test, preds)
    nasa_val = nasa_score(y_test, preds)

    results.append([model_name, rmse_val, mae_val, r2_val, nasa_val])
    predictions_dict[model_name] = preds

    save_prediction_scatter(y_test, preds, model_name, RESULTS_DIR)
    save_error_distribution(y_test, preds, model_name, RESULTS_DIR)


# SAVE RESULTS

comparison_df = pd.DataFrame(
    results,
    columns=["Model", "RMSE", "MAE", "R2", "NASA_Score"]
).sort_values("RMSE").reset_index(drop=True)

print("\n===== MODEL COMPARISON =====")
print(comparison_df.to_string(index=False))

comparison_df.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)

pred_df = pd.DataFrame({
    "Engine": engines,
    "True_RUL": y_test
})

for model_name, preds in predictions_dict.items():
    pred_df[f"{model_name}_Pred"] = preds

pred_df.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

display(comparison_df)
display(pred_df.head())

print("\nSaved result files:")
for f in RESULTS_DIR.glob("*"):
    print(f.name)


# DISPLAY SAVED PLOTS

plot_files = [
    "rul_distribution.png",
    "sensor_trend.png",
    "LSTM_loss.png",
    "GRU_loss.png",
    "CNN_loss.png",
    "LSTM_scatter.png",
    "GRU_scatter.png",
    "CNN_scatter.png",
    "LSTM_error.png",
    "GRU_error.png",
    "CNN_error.png",
]

for plot_name in plot_files:
    plot_path = RESULTS_DIR / plot_name
    if plot_path.exists():
        print(f"\nShowing: {plot_name}")
        img = plt.imread(plot_path)
        plt.figure(figsize=(8, 5))
        plt.imshow(img)
        plt.axis("off")
        plt.show()


# ZIP RESULTS

results_zip = ROOT_DIR / "results.zip"
with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in RESULTS_DIR.glob("*"):
        zf.write(f, arcname=f.name)

print("\nCreated results zip:", results_zip)
print("You can download it from Colab files panel.")


Using files:
Train: /content/data/train_FD001.txt
Test : /content/data/test_FD001.txt
RUL  : /content/data/RUL_FD001.txt

Train shape: (20631, 27)
Test shape : (13096, 27)
RUL shape  : (100, 1)


,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,125
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,125
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,125
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,125
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,125


,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735,125
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916,125
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166,125
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737,125
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130,125


,RUL_end
0,112
1,98
2,69
3,82
4,91



===== DATASET OVERVIEW =====
Shape: (20631, 27)
Number of Engines: 100
Dropping constant columns: ['op3', 's1', 's5', 's10', 's16', 's18', 's19']

X_train shape: (3586, 30, 17)
y_train shape: (3586,)
X_test shape : (100, 30, 17)
y_test shape : (100,)

===== Training LSTM =====
LSTM | Epoch 01/20 | Training Loss: 8679.6057
LSTM | Epoch 02/20 | Training Loss: 7512.5853
LSTM | Epoch 03/20 | Training Loss: 7216.0037
LSTM | Epoch 04/20 | Training Loss: 7325.7671
LSTM | Epoch 05/20 | Training Loss: 6588.5007
LSTM | Epoch 06/20 | Training Loss: 7411.4694
LSTM | Epoch 07/20 | Training Loss: 6909.6441
LSTM | Epoch 08/20 | Training Loss: 6723.7029
LSTM | Epoch 09/20 | Training Loss: 6662.2875
LSTM | Epoch 10/20 | Training Loss: 7284.7880
LSTM | Epoch 11/20 | Training Loss: 7157.3423
LSTM | Epoch 12/20 | Training Loss: 5521.9668
LSTM | Epoch 13/20 | Training Loss: 6915.6373
LSTM | Epoch 14/20 | Training Loss: 6060.1453
LSTM | Epoch 15/20 | Training Loss: 5966.3781
LSTM | Epoch 16/20 | Training L

,Model,RMSE,MAE,R2,NASA_Score
0,CNN,29.451734,21.934920,0.459853,13230.658696
1,GRU,67.189478,56.237018,-1.811207,70175.655428
2,LSTM,67.455556,56.462252,-1.833517,71986.293727


,Engine,True_RUL,LSTM_Pred,GRU_Pred,CNN_Pred
0,1,112,20.187881,20.518925,125.000000
1,2,98,20.187836,20.519003,102.101044
2,3,69,20.187712,20.519035,63.121178
3,4,82,20.187735,20.519035,68.085190
4,5,91,20.187780,20.519039,74.678444



Saved result files:
model_comparison.csv
LSTM_scatter.png
GRU_scatter.png
statistical_summary.csv
CNN_error.png
sensor_trend.png
GRU_loss.png
rul_distribution.png
test_predictions.csv
CNN_loss.png
GRU_error.png
CNN_scatter.png
LSTM_error.png
LSTM_loss.png

Showing: rul_distribution.png

Showing: sensor_trend.png

Showing: LSTM_loss.png

Showing: GRU_loss.png

Showing: CNN_loss.png

Showing: LSTM_scatter.png

Showing: GRU_scatter.png

Showing: CNN_scatter.png

Showing: LSTM_error.png

Showing: GRU_error.png

Showing: CNN_error.png

Created results zip: /content/results.zip
You can download it from Colab files panel.
